In [ ]:
pip install trl

# Downloading the dependencies

In [ ]:
import os
import re
import ast
import torch
import random
import warnings
import pandas as pd
from tqdm import tqdm
from peft import LoraConfig
from collections import Counter
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

In [ ]:
from huggingface_hub import login
login()

# Setting math helpers

In [ ]:
def is_allowed_expression(expression):
    if expression is None: 
        return False
        
    return re.fullmatch(r'[0-9+\-*/().\s]+', expression) is not None


def extract_numbers(expression):
    return [int(x) for x in re.findall(r'\d+', expression)]
    

def uses_numbers_correctly(expression, numbers):
    expr_counter = Counter(extract_numbers(expression))
    num_counter = Counter(numbers)
    
    for num, count in expr_counter.items():
        if count > num_counter[num]: 
            return False
    return True


def evaluate_expression(expression):
    if not is_allowed_expression(expression): 
        return None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            value = eval(expression, {'__builtins__': None}, {})
            if isinstance(value, (int, float)):
                return value
            return None
        except Exception:
            return None


def extract_pure_equation(row_data, numbers, target, eps=1e-9):
    raw_text = str(row_data)
    matches = re.findall(r'<answer>\s*(.*?)\s*</answer>', raw_text, re.DOTALL)
    
    if matches:
        expression = matches[-1].split('=')[0].strip()
        if is_allowed_expression(expression) and uses_numbers_correctly(expression, numbers):
            value = evaluate_expression(expression)
            
            if value is not None and abs(value - target) < eps:
                return True, expression

    math_chunks = re.findall(r'[0-9+\-*/().\s]+', raw_text)
    for chunk in math_chunks:
        parts = chunk.split('=') 
        for expression in parts:
            expression = expression.strip()
            if len(expression) < 3 or not is_allowed_expression(expression) or not uses_numbers_correctly(expression, numbers):
                continue

            value = evaluate_expression(expression)
            if value is not None and abs(value - target) < eps:
                return True, expression

    return False, None

# Dataset pipeline

In [ ]:
student_model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(student_model_id)

def format_for_student(example):
    messages = [
        {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"Numbers: {example['nums']}. Target: {example['target']}."},
        {"role": "assistant", "content": example['clean_equation']}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

In [ ]:
STUDENT_SYSTEM_PROMPT = """You are a precise arithmetic solver.
Using the given numbers exactly once, create an equation that equals the target.
Rules:
- Use only +, -, *, /
- Use each number at most once
- Return ONLY the final equation. Do NOT include explanations, words, or <think> tags."""

offline_teacher_data = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen2.5-7B-Instruct", split="train")

seen_puzzles = set()
perfect_training_rows = []

print("Extracting pure equations...")
for row in offline_teacher_data:
    nums = row["nums"]
    target = row["target"]
    is_valid, clean_equation = extract_pure_equation(row, nums, target)

    if is_valid:
        puzzle_key = (tuple(nums), target)
        if puzzle_key not in seen_puzzles:
            seen_puzzles.add(puzzle_key)

            perfect_training_rows.append({
                "nums": nums,
                "target": target,
                "clean_equation": clean_equation
            })

            if len(perfect_training_rows) >= 10000:
                break

print(f"Extracted {len(perfect_training_rows)} perfect training examples")

random.shuffle(perfect_training_rows)
split_idx = int(0.95 * len(perfect_training_rows))

train_dataset = Dataset.from_list(perfect_training_rows[:split_idx]).map(format_for_student)
value_dataset = Dataset.from_list(perfect_training_rows[split_idx:]).map(format_for_student)

print(f"Train size: {len(train_dataset)}, value size: {len(value_dataset)}")

# Downloading & training the student model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

training_config = SFTConfig(
    output_dir="./gemma_countdown_distilled",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,          
    logging_steps=25,
    eval_strategy="no",            
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_length=128,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_config,
    train_dataset=train_dataset.select_columns(["text"]),
    eval_dataset=value_dataset.select_columns(["text"]),
    peft_config=peft_config
)

trainer.train()

# Rejection Sampling (aka Best-of-N Inference)

In [ ]:
def pick_best_equation_sampled(model, tokenizer, nums, target, n_samples=32):
    model.eval()

    messages = [
        {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"Numbers: {nums}. Target: {target}."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            num_return_sequences=n_samples,
            pad_token_id=tokenizer.eos_token_id
        )

    for i in range(n_samples):
        new_tokens = outputs[i][input_len:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        is_valid, clean_eq = extract_pure_equation(generated_text, nums, target)
        
        if is_valid:
            return True, clean_eq

    fallback_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return False, fallback_text

In [ ]:
unseen_tasks = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "all", split="train")
test_subset = unseen_tasks.select(range(50))
correct_answers = 0

for i, task in enumerate(test_subset):
    nums = task["nums"]
    target = task["target"]

    is_valid, student_equation = pick_best_equation_sampled(trainer.model, tokenizer, nums, target, n_samples=32)
    correct_answers += int(is_valid)

    if i <= 5:
        print(f"Target: {target} | Nums: {nums}")
        print(f"Best Student Output: {student_equation}")
        print(f"Is Correct?: {is_valid}\n")

    else:
      break

print(f"=========================================")
print(f"Sampled Distillation Accuracy: {correct_answers / len(test_subset):.2%}")
print(f"=========================================")

# Submission

In [ ]:
test_csv_path = '/kaggle/input/datasets/weinedrive/test-public/test_public.csv'
df_test = pd.read_csv(test_csv_path)

def parse_nums(value):
    if isinstance(value, str):
        return ast.literal_eval(value) 
    return val

if 'nums' in df_test.columns:
    df_test['nums'] = df_test['nums'].apply(parse_nums)

ds_test = df_test.to_dict('records')
submission_rows = []

for index, expression in enumerate(tqdm(ds_test)):
    nums = expression["nums"]
    target = expression["target"]
    row_id = expression.get("id", index)

    is_valid, best_equation = pick_best_equation_sampled(
        trainer.model,
        tokenizer,
        nums,
        target,
        n_samples=32
    )

    submission_rows.append({
        "id": row_id,
        "equation": best_equation
    })

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)

print("\nSuccessfully saved submission.csv!")
display(submission_df.head())

# Submit to HuggingFace Hub

In [ ]:
repository_id = 'Dowlsa/gemma-countdown-distilled'
trainer.model.push_to_hub(repository_id)
tokenizer.push_to_hub(repository_id)